<a href="https://colab.research.google.com/github/yuvan1492/unsloth-llama-3-8b-bnb-4bit_training_template/blob/main/unsloth_llama_3_8b_bnb_4bit_training_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning Llama-3 8B with Unsloth on Google Colab

An optimized, memory-efficient pipeline for fine-tuning Large Language Models (LLMs) on a single consumer-grade or free-tier GPU. This project utilizes the **Unsloth** framework, **4-bit quantization**, and **LoRA (Low-Rank Adaptation)** to fine-tune the `Llama-3-8b-bnb-4bit` base model on the instruction-based Alpaca dataset using a Google Colab Tesla T4 instance.

---

## ⚡ Project Overview

Training modern large language models usually demands server-grade setups with massive VRAM (60GB+). This project demonstrates how to bypass hardware restrictions and train an 8-Billion parameter model natively inside a free Google Colab session containing a **Tesla T4 GPU (approx. 14.5 GB VRAM)**.

### Performance Highlights
* **Memory Optimization:** Reduces VRAM utilization by up to 70-80% using 4-bit model quantization and 8-bit AdamW optimizers.
* **Speed:** Leverages Unsloth's custom OpenAI Triton GPU kernels to train up to 2x–5x faster than native Hugging Face pipelines.
* **Cost Effective:** Achieves full training compatibility on 100% free cloud resources.

---

## 🛠️ Tech Stack & Core Libraries

* **Base Model:** `unsloth/llama-3-8b-bnb-4bit` (Meta Llama-3 pre-quantized configuration)
* **Dataset:** `yahma/alpaca-cleaned` (Standard clean instruction-following dataset)
* **Frameworks:** [Unsloth](https://github.com), PyTorch, Hugging Face (Transformers, PEFT, TRL)
* **Hardware Acceleration:** NVIDIA Tesla T4 GPU (Google Colab Environment)

---

## 📖 Step-by-Step Architecture

### 1. Environment & Memory Safeguards
The project begins by modifying PyTorch’s low-level memory handling structure (`expandable_segments:True`). This ensures that GPU memory is managed dynamically, avoiding the physical fragmentation crashes (`CUDA Out of Memory`) typical of large parameter loads.

### 2. 4-bit Quantization Load
Instead of loading the model weights in standard 16-bit high-precision floating points (which requires over 16GB just to sit idle), the model weights are loaded in **4-bit precision**. This compresses the static parameter arrays significantly so the engine fits entirely inside a ~12.5GB boundary, leaving vital overhead room for training variables.

### 3. LoRA Layer Injection
Instead of fine-tuning all 8 billion parameters, **Low-Rank Adaptation (LoRA)** locks the base weights in place. It attaches tiny, flexible mathematical adapter matrices (Rank `r=16`) to the primary attention layers (`q_proj`, `v_proj`, etc.). During training, only these lightweight adapters are modified.

### 4. Alpaca Prompt Mapping
Data is structured using the uniform Alpaca instruction template. Every training example is explicitly packed with structural text blocks (`### Instruction`, `### Input`, `### Response`) along with an End-Of-String token (`<|end_of_text|>`) to teach the model how to natively structure its thoughts and when to stop generating text.

### 5. Supervised Fine-Tuning Execution
The execution pipeline runs for a concise 30 steps using a low micro-batch layout (`per_device_train_batch_size = 2`) coupled with `gradient_accumulation_steps = 4` to simulate larger batch training runs without memory inflation. The `adamw_8bit` optimizer tracks learning adjustments with minimal data space.

---

## 🚀 How to Run this Project

1. Fork or clone this repository.
2. Upload the `.ipynb` file to your **Google Drive** or open it directly in **Google Colab**.
3. Change the runtime to a GPU: Go to **Runtime** > **Change runtime type** > select **GPU** (T4, L4, or A100) > Click **Save**.
4. Run the code cells sequentially.

---

## 📊 Key Configurations & Hyperparameters

| Parameter | Configuration | Purpose |
| :--- | :--- | :--- |
| **Max Sequence Length** | `1024` | Limits maximum context token parsing window |
| **Quantization Precision** | `4-bit` | Compresses target weights to fit within 15GB VRAM |
| **LoRA Rank (`r`)** | `16` | Size and depth of trainable adapter metrics |
| **LoRA Alpha** | `16` | Tuning multiplier determining adapter override strength |
| **Learning Rate** | `2e-4` | Size of steps taken by optimization gradients |
| **Optimizer** | `adamw_8bit` | Uses 8-bit quantization to conserve optimizer tracking space |

---

In [1]:
# Install unsloth directly from the package manager without using git
!pip install unsloth

# Ensure the required performance backends are ready
!pip install --no-deps xformers packaging ninja einops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199

In [2]:
from unsloth import FastLanguageModel
import torch
import gc
import os

# 1. Force environmental configurations to manage fragmented memory allocations
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. Aggressively purge and scrub all ghost allocations out of your current CUDA cache
if "model" in locals(): del model
if "tokenizer" in locals(): del tokenizer
gc.collect()
torch.cuda.empty_cache()

# 3. Load the 4-bit repository (Switching to 8B model to avoid OOM)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 1024,
    load_in_4bit = True,
    trust_remote_code = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.9.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [3]:
# Configure the training adapters using Unsloth's optimized wrapper
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                # Suggestions: 8, 16, 32, 64, 128 (higher uses slightly more VRAM)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,      # Heavily optimized to 0 for maximum speed
    bias = "none",         # Optimized to "none"
    use_gradient_checkpointing = "unsloth", # Crucial VRAM saver for T4 GPUs
    random_state = 3407,
    max_seq_length = 1024,
)

print("LoRA adapters successfully injected! VRAM footprint is optimized.")

Unsloth 2026.9.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA adapters successfully injected! VRAM footprint is optimized.


In [5]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Load the widely accessible, cleaned Alpaca instruction dataset
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

# 2. Define the structural template formatting for instruction tuning
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Crucial to signal to the model when to stop generating text

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Format the text block and append the end-of-string token
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Map the layout transformation array across your dataset pool
dataset = dataset.map(formatting_prompts_func, batched = True)

# 3. Configure the low-footprint SFTTrainer pipeline
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,     # Processes small micro-batches to fit T4 limits
        gradient_accumulation_steps = 4,     # Accumulates updates to preserve gradient stability
        warmup_steps = 5,
        max_steps = 30,                      # Keeps the initial checkpoint run brief
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",                # Conserves ~2GB of VRAM over traditional 32-bit optimizers
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 4. Start the training iterations
trainer_stats = trainer.train()


README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/51760 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,760 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.582700
2,2.095900
3,1.669510
4,1.866049
5,1.655432
6,1.481316
7,1.084072
8,1.264826
9,1.162833
10,1.101239


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.


In [6]:
# Enable native inference acceleration wrappers
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "List three distinct benefits of using quantization during large language model training.", # Instruction
        "",                                                                                         # Input (Leave blank if none)
        ""                                                                                          # Response (Leave blank for generation)
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

# Generate response tokens
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)


<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
List three distinct benefits of using quantization during large language model training.

### Input:


### Response:


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. **Reduced Computational Cost:** Quantization involves converting floating-point numbers to smaller, fixed-point numbers, which reduces the amount of memory and computation required for model training. This can significantly speed up the training process and allow for larger models to be trained with less hardware resources.

2. **Improved Model Performance:** Quantization can also improve model performance by reducing the impact of floating-point rounding errors. This can result in more accurate predictions and better generalization to new data.

3. **Better Model Interpretability:** Quantized models are often easier to interpret and understand, as they can be represented as a series of binary operations, which makes it easier to understand how the model arrives at its predictions. This can be especially useful for applications that require high levels of explainability.<|end_of_text|>
